# Assignment 2: Graph Analysis of Architectural Structure 🏗️
## Uffizi Gallery, Florence - Second Floor

**Objective** 🎯
Perform a graph-based analysis of the architectural structure, and interpret the results in terms of spatial organization.

### Tasks
1. **Compute Graph Metrics** 📊
Calculate at least three of the graph metrics discussed during the lectures:
*   Degree Centrality 🔗
*   Closeness Centrality 🌉
*   Shortest Paths 🛣️
*   *etc.*

2. **Interpret the Results** 💡
Provide a clear interpretation of the computed metrics in the context of the building:
*   Which spaces are the most connected? 🤝
*   Which elements act as critical connectors or bottlenecks? 🚪
*   Are there clusters or zones within the building? 🏘️
*   *etc.*

3. **Analyze Spatial Organization** 🧭
Explain what the graph reveals about the building’s spatial structure:
*   Circulation patterns 🚶‍♂️
*   Hierarchy of spaces 🏢
*   Accessibility and connectivity 🔄
*   Functional zoning 🗂️
*   *etc.*

### Deliverables 📦
*   **Graph Analysis Results**: Summaries or tables of computed metrics.
*   **Visualizations** 📈: Graphs, network diagrams, plots.
*   **Short Report** ✍️: Key findings from the analysis, emphasizing why graph analysis is useful for understanding architectural datasets.

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\giofo\Desktop\ARCHIVIO\MASTERS\MACAD\2_MACAD\0_MACAD DRIVE\AIA\Graph_ML-Giovanni_Carlo_Volpe\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.33) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [5]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Cnvert to Brep and Import the gallery floor plan

In [9]:
from topologicpy.Topology import Topology
from topologicpy.Cluster import Cluster
from topologicpy.Shell import Shell
from topologicpy.Face import Face

renderer = "vscode"

# 1. Import OBJ file
obj_path = r"Assignment_02_Giovanni-Carlo-Volpe_Uffizi-brep.obj"
obj = Topology.ByOBJPath(obj_path)
print("Imported objects:", len(obj), type(obj[0]))

# 2. Cluster and Merge
obj = Cluster.ByTopologies(obj)
obj = Topology.SelfMerge(obj)  
print("Merged OBJ floorplan:", obj)

# 3. Export and import BREP
brep_path = r"Assignment_02_Giovanni-Carlo-Volpe_Uffizi.brep"
Topology.ExportToBREP(obj, brep_path)
brep = Topology.ByBREPPath(brep_path)
print("Imported BREP floorplan:", brep)

# 4. Clean up BREP (this might take a few minutes)
triangles = Cluster.Faces(brep)

# Create a shell from the triangles
shell = Shell.ByFaces(triangles)

# Extract the external and internal boundaries of the shell
eb = Shell.ExternalBoundary(shell)
ib_list = Shell.InternalBoundaries(shell)

# Create a face from the external boundary and internal boundaries
floorplan = Face.ByWires(eb, ib_list)

# Remove collinear edges from the floorplan
floorplan = Topology.RemoveCollinearEdges(floorplan)
print("Cleaned floorplan Face:", floorplan)

# 5. Save the cleaned BREP
clean_brep_path = r"Assignment_02_Giovanni-Carlo-Volpe_Uffizi_face.brep"
Topology.ExportToBREP(floorplan, clean_brep_path)
print(f"Successfully exported clean face BREP to: {clean_brep_path}")

Imported objects: 2 <class 'topologic_core.Cluster'>
Merged OBJ floorplan: <topologic_core.Cluster object at 0x000002BDDA794FF0>
Topology.ExportToBREP - Error: a file already exists at the specified path and overwrite is set to False. Returning None.
Imported BREP floorplan: <topologic_core.Cluster object at 0x000002BDDA7F4C30>
Cleaned floorplan Face: <topologic_core.Face object at 0x000002BDB8826CF0>
Successfully exported clean face BREP to: Assignment_02_Giovanni-Carlo-Volpe_Uffizi_face.brep


In [10]:
# Visualize Geometry
Topology.Show(floorplan,
              camera=[0,0,6],
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)